In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# Jefferson Township – Monthly Panel Builder (EMS/Fire + NH + MF + Apt runs)
# Inputs (clean):
#   - nh_data_clean.csv
#   - fire_and_ems_runs_clean.csv  (must include 'ems' dummy: 1=EMS, 0=non‑EMS)
#   - parcels_jefferson_monthly_full.csv
#
# Output:
#   - panel_monthly_with_parcels.csv
# ─────────────────────────────────────────────────────────────────────────────

import pandas as pd, numpy as np
from pathlib import Path

# =========================== 0) Config / Paths ================================
ROOT       = Path().resolve().parents[0]
CLEAN_DIR  = ROOT / "data" / "clean"

NH_PATH      = CLEAN_DIR / "nh_data_clean.csv"
RUNS_PATH    = CLEAN_DIR / "fire_and_ems_runs_clean.csv"  # includes 'ems' dummy
PARCELS_PATH = CLEAN_DIR / "parcels_jefferson_monthly_full.csv"
OUT_PATH     = CLEAN_DIR / "panel_monthly_with_parcels.csv"

# Behavior flags
IMPUTE_INDUSTRIAL_AREA     = True   # impute industrial building area if 0/missing
IMPUTE_MIN_SHARE           = 10     # rows needed to build sqft-per-$ imputer

# Beds vs. runs weighting (beds reflect actual values; runs can be coverage-weighted)
TAYLOR_BEDS_WEIGHT = 1.00
TAYLOR_RUNS_WEIGHT = 0.20

# Densities (people/jobs per 1,000 sqft)
DENSITY_RES_PEOPLE_PER_1K_SQFT = 0.9
DENSITY_MF_PEOPLE_PER_1K_SQFT  = 1.4
DENSITY_COM_JOBS_PER_1K_SQFT   = 2.0
DENSITY_IND_JOBS_PER_1K_SQFT   = 1.0

# Codes / cues
LANDUSE_MF_CODE = 429           # Multifamily dwelling (parcels)
RUNS_APT_CODES  = {429}         # Multifamily dwelling (runs)
MF_CUES  = ["APART", "APT", "MULTI", "DUPLEX", "TRIPLEX", "TOWNHOME", "CONDO"]

# ============================ 1) Helpers =====================================
def monthify(dt_series):
    s = pd.to_datetime(dt_series, errors="coerce")
    return s.dt.to_period("M").dt.to_timestamp()

def enforce_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def any_contains_val(row, cols, cues):
    for c in cols:
        if c in row and pd.notna(row[c]) and any(k in str(row[c]).upper() for k in cues):
            return True
    return False

# ============================ 2) Nursing homes ================================
nh = pd.read_csv(NH_PATH, low_memory=False)
need_min = {"report_month","provider_name"}
missing = need_min - set(nh.columns)
if missing:
    raise ValueError(f"NH input missing columns: {missing}")

nh["provider_name"] = nh["provider_name"].astype(str).str.upper()
nh["report_month"]  = monthify(nh["report_month"])

# Pick best bed column
BED_CANDIDATES = [
    "number_of_certified_beds","total_number_of_beds","licensed_beds",
    "certified_beds","beds","number_of_beds"
]
bed_counts = {c: nh[c].notna().sum() for c in BED_CANDIDATES if c in nh.columns}
if not bed_counts:
    raise ValueError("No recognizable bed column found in nh_data_clean.csv.")
bed_col = max(bed_counts, key=bed_counts.get)
nh[bed_col] = pd.to_numeric(nh[bed_col], errors="coerce")
nh = nh.dropna(subset=["report_month", bed_col])

# Per-facility-month
fac_m = (nh.groupby(["provider_name","report_month"], as_index=False)
           .agg(beds=(bed_col,"max")))

FAC_SAGE   = "SAGE PARK"
FAC_TAYLOR = "TAYLOR SPRINGS"

fac_m["beds_sage_park_raw"] = np.where(fac_m["provider_name"].str.contains(FAC_SAGE),   fac_m["beds"], 0)
fac_m["beds_taylor_raw"]    = np.where(fac_m["provider_name"].str.contains(FAC_TAYLOR), fac_m["beds"], 0)

nh_monthly = (fac_m.groupby("report_month", as_index=False)
                .agg(beds_sage_park_raw=("beds_sage_park_raw","sum"),
                     beds_taylor_raw=("beds_taylor_raw","sum")))

# Beds: unweighted
nh_monthly["beds_sage_park"]       = nh_monthly["beds_sage_park_raw"]
nh_monthly["beds_taylor_springs"]  = nh_monthly["beds_taylor_raw"] * TAYLOR_BEDS_WEIGHT
nh_monthly["nh_total_certified_beds"] = nh_monthly["beds_sage_park"] + nh_monthly["beds_taylor_springs"]

# Fill continuous monthly index and ffill
if not nh_monthly.empty:
    full = pd.date_range(nh_monthly["report_month"].min(),
                         nh_monthly["report_month"].max(), freq="MS")
    nh_monthly = (nh_monthly.set_index("report_month")
                              .reindex(full)
                              .rename_axis("month")
                              .reset_index())
    cols_ff = ["beds_sage_park","beds_taylor_raw","beds_taylor_springs","nh_total_certified_beds"]
    nh_monthly[cols_ff] = nh_monthly[cols_ff].ffill()
else:
    nh_monthly = pd.DataFrame(columns=["month","beds_sage_park","beds_taylor_raw","beds_taylor_springs","nh_total_certified_beds"])

# ============================ 3) Runs & EMS/Fire ==============================
runs = pd.read_csv(RUNS_PATH, low_memory=False)
need = {"incident_date","incident_number","property_use_code","address","ems"}
missing = need - set(runs.columns)
if missing:
    raise ValueError(f"Runs input missing columns: {missing}")

runs["month"] = monthify(runs["incident_date"])
runs = runs.dropna(subset=["month"])
runs["ems"] = pd.to_numeric(runs["ems"], errors="coerce").fillna(0).astype(int)

# Address normalization (vectorized)
addr = runs["address"].astype(str).str.upper().str.strip()
addr = addr.str.replace(r"\s+", " ", regex=True)
for old, new in [
    (" AVENUE", " AVE"), (" AVE.", " AVE"),
    (" ROAD", " RD"),    (" RD.", " RD"),
    (" STREET", " ST"),  (" ST.", " ST"),
    (" DRIVE", " DR"),   (" DR.", " DR"),
    (" LANE", " LN"),    (" LN.", " LN"),
]:
    addr = addr.str.replace(old, new, regex=False)
runs["address_norm"] = addr

# Facility prefix matches
sage_regex   = r"^\s*5201\s+(?:E\s+)?MORSE\b"  # allow '5201 E MORSE'
taylor_regex = r"^\s*748\s+TAYLOR\b"
runs["to_sage_park"]  = runs["address_norm"].str.contains(sage_regex, regex=True, na=False).astype(int)
runs["to_taylor_raw"] = runs["address_norm"].str.contains(taylor_regex, regex=True, na=False).astype(int)

# Row-wise EMS/Fire + facility splits
runs["ems_call"]  = runs["ems"]
runs["fire_call"] = 1 - runs["ems_call"]

runs["sage_total"]   = runs["to_sage_park"]
runs["taylor_total"] = runs["to_taylor_raw"]
runs["sage_ems"]     = runs["sage_total"]   * runs["ems_call"]
runs["sage_fire"]    = runs["sage_total"]   * runs["fire_call"]
runs["taylor_ems"]   = runs["taylor_total"] * runs["ems_call"]
runs["taylor_fire"]  = runs["taylor_total"] * runs["fire_call"]

# Monthly totals (pure reductions)
monthly_total = (runs.groupby("month", as_index=False)
                      .agg(total_calls=("incident_number","count"),
                           ems_calls=("ems_call","sum"),
                           fire_calls=("fire_call","sum")))

# Nursing-home runs (raw + weighted Taylor)
nh_runs = (runs.groupby("month", as_index=False)
              .agg(
                  runs_sage_total       = ("sage_total","sum"),
                  runs_taylor_total_raw = ("taylor_total","sum"),
                  runs_sage_ems         = ("sage_ems","sum"),
                  runs_taylor_ems_raw   = ("taylor_ems","sum"),
              ))
nh_runs["runs_sage_fire"]       = nh_runs["runs_sage_total"]      - nh_runs["runs_sage_ems"]
nh_runs["runs_taylor_fire_raw"] = nh_runs["runs_taylor_total_raw"] - nh_runs["runs_taylor_ems_raw"]
nh_runs["runs_taylor_total"]    = (nh_runs["runs_taylor_total_raw"] * TAYLOR_RUNS_WEIGHT).round(0).astype(int)
nh_runs["runs_taylor_ems"]      = (nh_runs["runs_taylor_ems_raw"]   * TAYLOR_RUNS_WEIGHT).round(0).astype(int)
nh_runs["runs_taylor_fire"]     = nh_runs["runs_taylor_total"] - nh_runs["runs_taylor_ems"]

# Apartment runs — EXACT code match (429)
runs["property_use_code"] = pd.to_numeric(runs["property_use_code"], errors="coerce")
APT_CODE = 429  # Multifamily dwelling

runs["to_apartment"] = runs["property_use_code"].eq(APT_CODE).astype(int)
runs["apt_ems"]      = runs["to_apartment"] * runs["ems_call"]
runs["apt_fire"]     = runs["to_apartment"] * runs["fire_call"]

apt_runs = (runs.groupby("month", as_index=False)
                .agg(
                    runs_apartment_total=("to_apartment", "sum"),
                    runs_apartment_ems=("apt_ems", "sum"),
                    runs_apartment_fire=("apt_fire", "sum"),
                ))

# ============================ 4) Parcels & land use ===========================
parcels = pd.read_csv(PARCELS_PATH, low_memory=False)
date_col = "snapshot_month" if "snapshot_month" in parcels.columns else "report_month"
parcels["month"] = monthify(parcels[date_col])

num_cols = ["apprlnd","apprbld","apprtot","area_a","acrea","land_sqft","landuse"]
parcels = enforce_numeric(parcels, num_cols)

# land sqft convenience
if "land_sqft" in parcels.columns:
    parcels["land_sqft_use"] = parcels["land_sqft"]
elif "acrea" in parcels.columns:
    parcels["land_sqft_use"] = parcels["acrea"] * 43560
else:
    parcels["land_sqft_use"] = np.nan

# Base category from pclass
PCLASS_MAP = {
    "R": "1 Residential",
    "A": "2 Agricultural",
    "M": "3 Mineral",
    "C": "4 Commercial",
    "I": "5 Industrial",
    "U": "6 Public Utility Real",
    "P": "7 Public Utility Personal",
    "E": "8 General Personal",
    "Z": "8 General Personal"
}
parcels["pclass"] = parcels["pclass"].astype(str).str.upper()
parcels["auditor_category"] = parcels["pclass"].map(PCLASS_MAP).fillna("8 General Personal")

# Impute industrial building area (area_a_used)
if IMPUTE_INDUSTRIAL_AREA:
    src  = parcels.copy()
    comp = src[src["auditor_category"].isin(["4 Commercial","1 Residential"])].copy()
    comp = comp[(comp["area_a"].fillna(0) > 0) & (comp["apprbld"].fillna(0) > 0)]
    if len(comp) >= IMPUTE_MIN_SHARE:
        comp["sqft_per_dollar"] = comp["area_a"] / comp["apprbld"]
        med = (comp.groupby("auditor_category", as_index=False)["sqft_per_dollar"]
                  .median().sort_values("sqft_per_dollar", ascending=False))
        if "4 Commercial" in med["auditor_category"].values:
            sqft_per_dollar = float(med.loc[med["auditor_category"]=="4 Commercial","sqft_per_dollar"].iloc[0])
        elif "1 Residential" in med["auditor_category"].values:
            sqft_per_dollar = float(med.loc[med["auditor_category"]=="1 Residential","sqft_per_dollar"].iloc[0])
        else:
            sqft_per_dollar = np.nan
    else:
        sqft_per_dollar = np.nan

    parcels["area_a_imputed"] = np.nan
    mask_ind = parcels["auditor_category"].eq("5 Industrial") & (parcels["area_a"].fillna(0) == 0)
    if not np.isnan(sqft_per_dollar):
        can_use_dollar = mask_ind & parcels["apprbld"].notna() & (parcels["apprbld"] > 0)
        parcels.loc[can_use_dollar, "area_a_imputed"] = parcels.loc[can_use_dollar, "apprbld"] * sqft_per_dollar
    fallback = mask_ind & parcels["land_sqft_use"].notna() & parcels["area_a_imputed"].isna()
    parcels.loc[fallback, "area_a_imputed"] = parcels.loc[fallback, "land_sqft_use"]
    parcels["area_a_used"] = np.where(mask_ind & parcels["area_a_imputed"].notna(),
                                      parcels["area_a_imputed"], parcels["area_a"])
else:
    parcels["area_a_used"] = parcels["area_a"]

# ── Multifamily detection (exact code + text backup on your real columns) ────
PARCEL_TEXT_COLS = ["descr1", "descr2", "descr3", "landuse", "proptyp"]

is_mf_code = parcels["landuse"].eq(LANDUSE_MF_CODE)
is_mf_text = parcels.apply(lambda r: any_contains_val(r, PARCEL_TEXT_COLS, MF_CUES), axis=1)

parcels["res_subtype"] = np.where(
    parcels["auditor_category"].eq("1 Residential") & (is_mf_code | is_mf_text),
    "res_mf",
    np.where(parcels["auditor_category"].eq("1 Residential"), "res_sf", "")
)

# Compute sqft by subtype
parcels["res_sf_area_sqft"] = np.where(
    (parcels["auditor_category"].eq("1 Residential")) & (parcels["res_subtype"]=="res_sf"),
    parcels["area_a_used"], 0.0
)
parcels["res_mf_area_sqft"] = np.where(
    (parcels["auditor_category"].eq("1 Residential")) & (parcels["res_subtype"]=="res_mf"),
    parcels["area_a_used"], 0.0
)

# Commercial (remove any MF-like records that slipped into commercial)
parcels["com_area_sqft_raw"] = np.where(parcels["auditor_category"].eq("4 Commercial"),
                                       parcels["area_a_used"], 0.0)
suspect_mf_in_com = is_mf_code | parcels.apply(
    lambda r: any_contains_val(r, PARCEL_TEXT_COLS, MF_CUES), axis=1
)
parcels["com_area_sqft_no_mf"] = np.where(suspect_mf_in_com, 0.0, parcels["com_area_sqft_raw"])

# Industrial
parcels["ind_area_sqft"] = np.where(parcels["auditor_category"].eq("5 Industrial"),
                                    parcels["area_a_used"], 0.0)

# Monthly aggregates
agg_cols = {
    "res_sf_area_sqft":"sum",
    "res_mf_area_sqft":"sum",
    "com_area_sqft_no_mf":"sum",
    "ind_area_sqft":"sum",
    "apprbld":"sum",
    "apprtot":"sum"
}
parcel_m = (parcels.groupby("month", as_index=False)
                   .agg(agg_cols)
                   .rename(columns={"com_area_sqft_no_mf":"com_area_sqft"}))

# ============================ 5) Population / Worker proxies ==================
parcel_m["est_residents_sf"]     = (parcel_m["res_sf_area_sqft"] / 1_000.0) * DENSITY_RES_PEOPLE_PER_1K_SQFT
parcel_m["est_residents_mf"]     = (parcel_m["res_mf_area_sqft"] / 1_000.0) * DENSITY_MF_PEOPLE_PER_1K_SQFT
parcel_m["est_residents_total"]  = parcel_m["est_residents_sf"] + parcel_m["est_residents_mf"]

parcel_m["est_workers_com"]      = (parcel_m["com_area_sqft"] / 1_000.0) * DENSITY_COM_JOBS_PER_1K_SQFT
parcel_m["est_workers_ind"]      = (parcel_m["ind_area_sqft"] / 1_000.0) * DENSITY_IND_JOBS_PER_1K_SQFT
parcel_m["est_workers_total"]    = parcel_m["est_workers_com"] + parcel_m["est_workers_ind"]

# ============================ 6) Merge panel =================================
panel = (monthly_total
         .merge(nh_runs, on="month", how="left")
         .merge(nh_monthly, on="month", how="left")
         .merge(parcel_m, on="month", how="left")
         .merge(apt_runs, on="month", how="left")   # apartment runs by code
         .sort_values("month")
         .reset_index(drop=True))

# Scaled convenience fields (per 1k sqft; per $1M)
for c in ["res_sf_area_sqft","res_mf_area_sqft","com_area_sqft","ind_area_sqft"]:
    panel[f"{c}_k"] = panel[c] / 1_000.0
for c in ["apprbld","apprtot"]:
    panel[f"{c}_M"] = panel[c] / 1_000_000.0

# Optional lags & YoY deltas
ADD_LAGS = True
ADD_YOY  = True

if ADD_LAGS:
    for c in ["res_sf_area_sqft_k","res_mf_area_sqft_k","com_area_sqft_k","ind_area_sqft_k",
              "apprbld_M","apprtot_M",
              "est_residents_total","est_workers_total",
              "beds_sage_park","beds_taylor_springs","nh_total_certified_beds",
              "ems_calls","fire_calls","total_calls",
              "runs_apartment_total","runs_apartment_ems","runs_apartment_fire"]:
        if c in panel.columns:
            panel[f"{c}_lag6"] = panel[c].shift(6)

if ADD_YOY:
    for c in ["res_sf_area_sqft_k","res_mf_area_sqft_k","com_area_sqft_k","ind_area_sqft_k",
              "apprbld_M","apprtot_M",
              "est_residents_total","est_workers_total",
              "beds_sage_park","beds_taylor_springs","nh_total_certified_beds",
              "ems_calls","fire_calls","total_calls",
              "runs_apartment_total","runs_apartment_ems","runs_apartment_fire"]:
        if c in panel.columns:
            panel[f"{c}_yoy"] = panel[c] - panel[c].shift(12)

# ============================ 7) Save ========================================
panel.to_csv(OUT_PATH, index=False)
print(f"Saved monthly panel to: {OUT_PATH}")
print(f"Rows: {len(panel):,}  |  Columns: {len(panel.columns):,}")
keep_cols = ("^month$|ems_calls$|fire_calls$|runs_.*|beds_.*|res_.*_k$|com_area_sqft_k$|ind_area_sqft_k$|"
             "est_residents_total$|est_workers_total$|runs_apartment_.*")
print(panel.filter(regex=keep_cols).head(8))

Saved monthly panel to: C:\Repositories\jefferson-township-run-forecasting\data\clean\panel_monthly_with_parcels.csv
Rows: 84  |  Columns: 73
       month  ems_calls  fire_calls  runs_sage_total  runs_taylor_total_raw  \
0 2018-08-01        115          91               10                      0   
1 2018-09-01        126          90                6                      0   
2 2018-10-01        123          82                7                      0   
3 2018-11-01        120          85                2                      0   
4 2018-12-01        116          86                5                      0   
5 2019-01-01        100          99                4                      0   
6 2019-02-01         92          76                3                      0   
7 2019-03-01        113          76                3                      0   

   runs_sage_ems  runs_taylor_ems_raw  runs_sage_fire  runs_taylor_fire_raw  \
0             10                    0               0              